In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

project = Path.cwd().parent
file = project / "data" / "raw" / "FreshBasket_Loyalty_Churn_Dataset.xlsx"

profile = pd.read_excel(file, sheet_name="fb Customer Profile", header=1)
activity = pd.read_excel(file, sheet_name="fb Monthly Activity", header=1)
churn = pd.read_excel(file, sheet_name="fb Churn Label", header=1)

profile.columns = profile.columns.str.strip().str.upper().str.replace(" ", "_")
activity.columns = activity.columns.str.strip().str.upper().str.replace(" ", "_")
churn.columns = churn.columns.str.strip().str.upper().str.replace(" ", "_")

profile = profile.dropna(how="all").drop_duplicates().reset_index(drop=True)
activity = activity.dropna(how="all").drop_duplicates().reset_index(drop=True)
churn = churn.dropna(how="all").drop_duplicates().reset_index(drop=True)

profile["CUSTOMER_ID"] = pd.to_numeric(profile["CUSTOMER_ID"], errors="coerce").astype("Int64")
activity["CUSTOMER_ID"] = pd.to_numeric(activity["CUSTOMER_ID"], errors="coerce").astype("Int64")
churn["CUSTOMER_ID"] = pd.to_numeric(churn["CUSTOMER_ID"], errors="coerce").astype("Int64")

profile["SIGNUP_DATE"] = pd.to_datetime(profile["SIGNUP_DATE"], errors="coerce")
activity["MONTH"] = pd.to_datetime(activity["MONTH"], errors="coerce")
churn["OBSERVATION_END_DATE"] = pd.to_datetime(churn["OBSERVATION_END_DATE"], errors="coerce")
churn["LAST_PURCHASE_DATE"] = pd.to_datetime(churn["LAST_PURCHASE_DATE"], errors="coerce")

for col in ["AGE", "MARKETING_OPT_IN", "PREFERRED_STORE_ID", "HOME_STORE_PRICE_TIER"]:
    profile[col] = pd.to_numeric(profile[col], errors="coerce")

for col in [
    "TRANSACTIONS", "TOTAL_SPEND", "AVG_BASKET_VALUE",
    "DISTINCT_CATEGORIES", "PROMO_TXN_PCT", "APP_SESSIONS",
    "EMAILS_OPENED", "COUPONS_REDEEMED", "SUPPORT_TICKETS",
    "COMPLAINT_FLAG"
]:
    activity[col] = pd.to_numeric(activity[col], errors="coerce")

for col in ["MONTHS_OBSERVED", "INSUFFICIENT_HISTORY_FLAG", "CHURNED"]:
    churn[col] = pd.to_numeric(churn[col], errors="coerce")

for col in ["GENDER", "CITY", "MEMBERSHIP_TIER", "PREFERRED_CATEGORY"]:
    profile[col] = profile[col].astype("string").str.strip()

profile["GENDER"] = profile["GENDER"].str.upper()
profile["CITY"] = profile["CITY"].str.title()
profile["MEMBERSHIP_TIER"] = profile["MEMBERSHIP_TIER"].str.title()
profile["PREFERRED_CATEGORY"] = profile["PREFERRED_CATEGORY"].str.title()

profile_duplicates = profile["CUSTOMER_ID"].duplicated().sum()
activity_duplicates = activity.duplicated(["CUSTOMER_ID", "MONTH"]).sum()
churn_duplicates = churn["CUSTOMER_ID"].duplicated().sum()

negative_spend = (activity["TOTAL_SPEND"] < 0).sum()
negative_transactions = (activity["TRANSACTIONS"] < 0).sum()
invalid_basket = (activity["AVG_BASKET_VALUE"] < 0).sum()

zero_transactions = (activity["TRANSACTIONS"] == 0).sum()
zero_transactions_with_spend = (
    (activity["TRANSACTIONS"] == 0) &
    (activity["TOTAL_SPEND"] != 0)
).sum()

profile_ids = set(profile["CUSTOMER_ID"].dropna())
activity_ids = set(activity["CUSTOMER_ID"].dropna())
churn_ids = set(churn["CUSTOMER_ID"].dropna())

activity_without_profile = activity_ids - profile_ids
churn_without_profile = churn_ids - profile_ids

activity_months = activity["MONTH"].dropna().dt.to_period("M").sort_values().unique()

gaps = []

for customer_id, group in activity.groupby("CUSTOMER_ID"):
    months = group["MONTH"].dropna().dt.to_period("M").sort_values().unique()

    if len(months) > 1:
        expected = pd.period_range(months.min(), months.max(), freq="M")
        missing = expected.difference(months)

        if len(missing):
            gaps.append({
                "CUSTOMER_ID": customer_id,
                "MISSING_MONTHS": len(missing)
            })

gaps = pd.DataFrame(gaps)

outliers = {}

for col in [
    "TOTAL_SPEND",
    "AVG_BASKET_VALUE",
    "TRANSACTIONS",
    "APP_SESSIONS",
    "EMAILS_OPENED",
    "COUPONS_REDEEMED",
    "SUPPORT_TICKETS"
]:
    q1 = activity[col].quantile(0.25)
    q3 = activity[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers[col] = int(((activity[col] < lower) | (activity[col] > upper)).sum())

(project / "data" / "processed").mkdir(parents=True, exist_ok=True)
(project / "outputs").mkdir(parents=True, exist_ok=True)

profile.to_csv(project / "data" / "processed" / "customer_profile_clean.csv", index=False)
activity.to_csv(project / "data" / "processed" / "monthly_activity_clean.csv", index=False)
churn.to_csv(project / "data" / "processed" / "churn_label_clean.csv", index=False)
gaps.to_csv(project / "outputs" / "customer_month_gaps.csv", index=False)

quality = pd.DataFrame({
    "Check": [
        "Profile rows",
        "Activity rows",
        "Churn rows",
        "Duplicate profile customers",
        "Duplicate customer-months",
        "Duplicate churn customers",
        "Negative spend rows",
        "Negative transaction rows",
        "Negative basket values",
        "Zero transaction rows",
        "Zero transactions with non-zero spend",
        "Activity customers missing from profile",
        "Churn customers missing from profile",
        "Customers with internal month gaps",
        "Unique activity months"
    ],
    "Result": [
        len(profile),
        len(activity),
        len(churn),
        profile_duplicates,
        activity_duplicates,
        churn_duplicates,
        negative_spend,
        negative_transactions,
        invalid_basket,
        zero_transactions,
        zero_transactions_with_spend,
        len(activity_without_profile),
        len(churn_without_profile),
        len(gaps),
        len(activity_months)
    ]
})

quality

,Check,Result
0,Profile rows,2600
1,Activity rows,28015
2,Churn rows,2600
3,Duplicate profile customers,0
4,Duplicate customer-months,2
5,Duplicate churn customers,0
6,Negative spend rows,24
7,Negative transaction rows,0
8,Negative basket values,0
9,Zero transaction rows,2330
